# 🔑 Kunci Jawaban — Kuis Bab 9 (Fondasi LLM)

> **Buka file ini HANYA SETELAH** mencoba kuis sendiri. Membaca jawaban
> sebelum berjuang = nol pembelajaran (ilusi kompetensi).

Untuk tiap soal: jawaban → **kenapa** → kode referensi. Fokus belajar ada di bagian *kenapa*.


## Bagian A — Pilihan Ganda


### Soal 1 — Jawaban: **c) karakter/subword selalu tersedia sebagai fallback**

Kekuatan inti subword: vocabulary terbatas (mis. 50k merge) TAPI teks apa pun
bisa di-encode — kata yang belum pernah dilihat dipecah ke potongan yang
semuanya ada di vocabulary. Model tetap bisa menebak makna dari potongannya
(`unbeliev` + `able`), dan output decode balik TANPA kehilangan huruf.

Kenapa opsi lain menjebak:

- **a)** vocabulary subword justru sering LEBIH kecil dari word-based, tapi itu
  efek samping — bukan alasan keberadaannya.
- **b)** kata langka sering justru paling informatif (nama, istilah teknis).
- **d)** `<UNK>` membuat loss tak terukur untuk kata yang di-UNK-kan — itu
  gejala yang mau dihindari, bukan alasan.

Bukti lab Bagian 1: `encode_kata('newestt')` (typo) tetap menghasilkan simbol
yang semuanya ada di vocabulary — fallback karakter selalu ada.


### Soal 2 — Jawaban: **b) varians skor tumbuh ~d_k → softmax satu-hot → gradien mati**

Komponen Q, K iid dengan varians ~1 → setiap entri `Q·K` adalah penjumlahan
d_k produk acak → varians skor ~ d_k. Tanpa scaling, softmax pada vektor
varians besar = praktis satu-hot: probabilitas max ≈ 1, lainnya ≈ 0.
Gradien softmax juga ≈ 0 → sinyal belajar mati. `sqrt(d_k)` menormalkan
varians kembali ke ~1.

Kenapa opsi lain salah:

- **a)** softmax valid untuk skor negatif — masalahnya varians, bukan tanda.
- **c)** jarak perhatian ditentukan data & masking, bukan scaling.
- **d)** bukan konvensi: di lab Bagian 5.2, max prob tanpa scaling jauh lebih
  ekstrem, dan makin parah saat d_k dinaikkan.


### Soal 3 — Jawaban: **b) tanpa mask model melihat masa depan — loss curang**

Decoder dilatih memprediksi token t+1 dari token ≤ t. Tanpa causal mask,
attention di posisi t juga mengintip token t+1, t+2, ... — soal uji bocor ke
jawaban. Loss training tampak bagus, tapi saat inference (generate autoregresif)
token masa depan TIDAK ADA → model tidak pernah belajar situasi itu →
generasi hancur.

Kenapa opsi lain salah:

- **a)** softmax tetap valid; mask bekerja sebelum softmax (skor -1e9).
- **c)** jumlah parameter tak berubah karena mask.
- **d)** PE independen dari masking — ia menyuntik posisi, bukan visibilitas.

Bukti lab Bagian 5.3: baris 0 attention ber-mask = [1, 0, 0, 0]; baris i hanya
punya massa probabilitas pada kolom j ≤ i.


### Soal 4 — Jawaban: **b) PE_p · PE_q = f(p−q) — relative position dari dot product**

Identitas trigonometri: `sin p sin q + cos p cos q = cos((p−q)·f)`,
dijumlahkan per frekuensi → dot product dua baris PE hanya fungsi dari
SELISIH posisi. Itu bukan kebetulan desain — ini yang memungkinkan model
membaca 'seberapa berjauhan' dua token tanpa tabel posisi tambahan.

Kenapa opsi lain salah:

- **a)** PE = fungsi murni posisi; kalau bergantung token, ia bukan lagi
  'positional' encoding.
- **c)** sin/cos bounded [-1, 1] — justru supaya bisa extend ke posisi jauh.
- **d)** versi sinusoidal FIX (tidak dipelajari); versi learnable ada (GPT-2),
  tapi soal ini tentang varian sinusoidal.

Bukti lab Bagian 4.2: dot(jarak 3) sama persis di posisi (0,3), (5,8), (2,5).


### Soal 5 — Jawaban: **b) distribusi lebih datar — lebih variatif, risiko naik**

`softmax(logits / T)`: T besar mengecilkan selisih logits → distribusi mendekati
uniform → token langka makin sering terpilih. T kecil → distribusi tajam →
mendekati greedy. Di lab Bagian 7.2: max prob T=0.5 → 0.742 vs T=2.0 → 0.246.

Kenapa opsi lain salah:

- **a)** itu efek T DITURUNKAN.
- **c)** T mengubah distribusi sampling — jumlah token output diatur terpisah
  (max_tokens).
- **d)** tidak ada token yang dihapus oleh T; penyaringan itu kerja top-k/top-p.


### Soal 6 — Jawaban: **b) encoder-only (BERT-style)**

Klasifikasi satu kalimat = task 'memahami' → encoder bidirectional ideal:
tiap token melihat seluruh kalimat, [CLS]/pooling jadi fitur klasifikasi,
modelnya kecil → murah & cepat. Decoder-only bisa saja (dan sering dipakai
via API LLM), tapi untuk volume tinggi dengan budget kecil itu boros —
kamu membayar generasi token untuk yang bisa diselesaikan satu forward pass.

Kenapa opsi lain salah:

- **a)** generate-then-parse: lebih lambat, lebih mahal, dan menambah
  kegagalan parsing — tanpa untung akurasi untuk task sederhana.
- **c)** 'selalu paling akurat' tidak pernah benar; T5 unggul di
  text-to-text, bukan otomatis di klasifikasi murah.
- **d)** CNN bisa untuk teks pendek, tapi 'bukan model bahasa' salah —
  dan BERT-style memberi baseline yang lebih kuat.

Aturan praktis: pahami teks → encoder; hasilkan teks → decoder;
transformasi terstruktur input→output → encoder-decoder.


## Bagian B — Coding (kode referensi)


### Soal 7 — `softmax_stabil`

Dua kunci: (1) kurangi max per-baris — `exp` tetap finite untuk skor besar;
(2) normalisasi per-baris. `softmax([0, ln 3]) = [1/4, 3/4]` karena
exp(0) = 1 dan exp(ln 3) = 3, total 4.


In [ ]:
import numpy as np

def softmax_stabil(x, axis=-1):
    x = np.asarray(x, dtype=float)
    z = x - np.max(x, axis=axis, keepdims=True)
    e = np.exp(z)
    return e / np.sum(e, axis=axis, keepdims=True)

print(softmax_stabil(np.array([0.0, np.log(3.0)])))   # [0.25 0.75]
x = np.random.default_rng(0).normal(0, 30, (5, 7))
print(softmax_stabil(x).sum(axis=-1).round(9))        # tiap baris = 1


### Soal 8 — `attention`

Tiga langkah: skor berskala → softmax per baris → kalikan ke V.
Jebakan paling umum: lupa `sqrt(d_k)` — hasilnya masih 'berjalan' tapi
distribusi jauh lebih tajam, dan grader kuis ini menangkapnya.


In [ ]:
def attention(Q, K, V):
    d_k = Q.shape[-1]
    scores = Q @ K.T / np.sqrt(d_k)
    attn = softmax_stabil(scores, axis=-1)
    return attn @ V, attn

np.random.seed(9)
Q = np.random.randn(4, 8); K = np.random.randn(4, 8); V = np.random.randn(4, 8)
out, attn = attention(Q, K, V)
print(attn.sum(axis=-1).round(6))   # [1. 1. 1. 1.]
print(np.allclose(out, attn @ V))   # True - output = campuran berbobot V


### Soal 9 — `apply_causal_mask`

Mask = segitiga atas TANPA diagonal (k=1): posisi i boleh melihat dirinya
sendiri (j = i) tapi tidak token setelahnya. Menambahkan -1e9 (bukan
mengganti/menghapus baris) menjaga bentuk matrix sehingga softmax tetap
bisa dihitung per baris.


In [ ]:
def apply_causal_mask(scores):
    scores = np.asarray(scores, dtype=float)
    n = scores.shape[0]
    mask = np.triu(np.ones((n, n)), k=1)
    return scores + mask * -1e9

print(apply_causal_mask(np.array([[1., 2.], [3., 4.]])))
# [[   1. -1e+09]
#  [   3.    4.]]


### Soal 10 — `sample_topk`

Dua langkah: (1) seleksi — `argsort` menurun, ambil k pertama;
(2) sampling — softmax LOKAL pada k kandidat itu (probabilitas dinormalisasi
di dalam kandidat, bukan di seluruh vocabulary), lalu `rng.choice(idx, p=p)`.
Top-k memotong EKOR distribusi: token langka keluar dari kandidat meski
temperaturnya tinggi.


In [ ]:
def sample_topk(logits, k, rng):
    logits = np.asarray(logits, dtype=float)
    idx = np.argsort(logits)[::-1][:k]
    p = softmax_stabil(logits[idx])
    return int(rng.choice(idx, p=p))

lg = np.array([0.1, 2.2, 0.5, 1.3, 0.1, 0.9, 0.2, 0.4])
r = np.random.default_rng(9)
print({sample_topk(lg, 2, r) for _ in range(200)})   # {1, 3} saja
print({sample_topk(lg, 3, r) for _ in range(500)})   # subset {1, 3, 5}


---

Skor penuh? Lanjut ke `project-starter-tokenizer-mini/` — di sana semua konsep ini
disatukan menjadi tokenizer + mini-LM end-to-end dengan mode TDD.
